<a href="https://colab.research.google.com/github/roy-tokyo/aiagent/blob/main/sample_rag_application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#00- pip install 共通

In [5]:
!pip install -q requests
!pip install -q openai langsmith langchain-openai langchain-core langchain-community google-search-results --break-system-packages

#01-簡単なRAGを構築してみます。

In [6]:
!pip install -q langchain-community GitPython


## Document Loader

Githubに公開されているLangChainのドキュメントの一部を読み込みする。

LangChainのGitHubリポジトリからMarkdownファイルのみをクローンして読み込み、その件数を表示します。

In [9]:
from langchain_community.document_loaders import GitLoader

# ファイルフィルター関数: Markdownファイルのみを読み込み対象にする
def file_filter(file_path: str) -> bool:
    """
    指定されたファイルパスがMarkdownファイル(.md)かどうかを判定

    Args:
        file_path: チェックするファイルのパス

    Returns:
        bool: Markdownファイルの場合True、それ以外False
    """
    return file_path.endswith(".md")

# GitLoaderの初期化
loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",  # クローン元のGitHubリポジトリURL
    repo_path="./langchain",  # ローカルにクローンする保存先パス
    branch="master",  # 対象ブランチ
    file_filter=file_filter,  # Markdownファイルのみをフィルタリング
)

# ドキュメントの読み込みを実行
raw_docs = loader.load()

# 読み込まれたドキュメントの数を出力
print(len(raw_docs))

28


##Document transformer

In [4]:
!pip install -q langchain-text-splitters

前のステップで読み込んだMarkdownドキュメント（raw_docs）を、1455文字ごとのチャンクに分割します。chunk_overlap=0のため、チャンク間に重複はありません。分割後の総チャンク数が出力されます。

###chunk_sizeとchunk_overlapの一般的な設定について:
chunk_size（チャンクサイズ）
一般的な設定範囲: 500〜2000文字（または200〜1000トークン）
用途別の推奨値:

小さめ (200-500文字)

精密な検索が必要な場合
FAQ、短い質問応答
具体的な情報を素早く見つけたい場合


中程度 (500-1500文字) ⭐ 最も一般的

バランスの取れた選択
技術文書、ブログ記事
RAG（検索拡張生成）の標準的な用途


大きめ (1500-3000文字)

文脈が重要な場合
長い段落や章全体を保持したい場合
LLMのコンテキストウィンドウに余裕がある場合



chunk_overlap（オーバーラップ）
一般的な設定: chunk_sizeの10〜20%
推奨値:

chunk_size=1000 → chunk_overlap=100-200
chunk_size=1500 → chunk_overlap=150-300
chunk_size=500 → chunk_overlap=50-100

オーバーラップの目的:

文脈の連続性を保つ: チャンク境界で情報が分断されるのを防ぐ
検索精度の向上: 重要な情報がチャンク境界に跨る場合でも取得できる

In [10]:
from langchain_text_splitters import CharacterTextSplitter

# テキスト分割器の初期化
text_splitter = CharacterTextSplitter(
    chunk_size=1455,      # 1つのチャンクの最大文字数
    chunk_overlap=0       # チャンク間のオーバーラップ文字数（0=重複なし）
)

# ドキュメントを指定されたサイズのチャンクに分割
docs = text_splitter.split_documents(raw_docs)

# 分割後のドキュメント数を出力
print(len(docs))

69


In [11]:
from langchain_core import embeddings
from langchain_openai import OpenAIEmbeddings
import os
from google.colab import userdata

# Google ColabのシークレットからOpenAI APIキーを取得して環境変数に設定
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# OpenAIのEmbeddingモデルを初期化
# text-embedding-3-smallは軽量で高速なモデル（1536次元）
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# ベクトル化したいクエリテキスト
query = "Langchainを利用する理由は何ですか？"

# クエリテキストをベクトル（数値の配列）に変換
vector = embeddings.embed_query(query)

# ベクトルの次元数を出力（text-embedding-3-smallは1536次元）
print(len(vector))

# ベクトルの内容を出力（1536個の浮動小数点数のリスト）
print(vector)


1536
[-0.016815185546875, -0.0225372314453125, 0.03460693359375, -0.01142120361328125, 0.0394287109375, -0.01447296142578125, -0.07373046875, 0.03900146484375, -0.01558685302734375, -0.0364990234375, -0.01751708984375, 0.06622314453125, -0.030853271484375, 0.039642333984375, -0.01262664794921875, -0.0213165283203125, -0.01465606689453125, -0.0004718303680419922, 6.961822509765625e-05, 0.0301513671875, -0.035369873046875, 0.021270751953125, -0.035736083984375, 0.0132598876953125, -0.01690673828125, -0.01467132568359375, 0.030914306640625, 0.0095977783203125, 0.0243988037109375, -0.056640625, -0.032623291015625, -0.0457763671875, 0.0096893310546875, -0.012298583984375, 0.00870513916015625, -0.006786346435546875, -0.017608642578125, -0.0119781494140625, 0.0055999755859375, -0.0007848739624023438, 0.0233154296875, 0.02044677734375, 0.014434814453125, -0.0047454833984375, 0.0005350112915039062, 0.01023101806640625, -0.028839111328125, 0.00572967529296875, -0.0243682861328125, 0.025711059570

このコードの役割:

1. **Embedding（埋め込み）の生成**: テキストを数値ベクトルに変換
2. **意味の数値化**: 似た意味を持つテキストは近いベクトルになる
3. **検索の準備**: このベクトルを使って、意味的に類似したドキュメントを検索できる

In [12]:
!pip install  -q --break-system-packages \
  opentelemetry-api==1.38.0 \
  opentelemetry-sdk==1.38.0 \
  opentelemetry-proto==1.38.0 \
  opentelemetry-exporter-otlp-proto-common==1.38.0 \
  opentelemetry-exporter-otlp-proto-grpc==1.38.0

!pip install -q langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.8 MB/s eta 0:00:00


In [13]:
from langchain_chroma import Chroma

# Chromaベクトルデータベースを作成
# docs: 分割されたドキュメント
# embeddings: OpenAIのEmbeddingモデル
# ドキュメントを自動的にベクトル化してデータベースに保存
db = Chroma.from_documents(docs, embeddings)

# データベースをRetriever（検索器）として使用できるように変換
retriever = db.as_retriever()

# 検索クエリ（質問）
query = "Langchainを利用する理由は何ですか？"

# クエリに意味的に類似したドキュメントを検索
# デフォルトでは上位4件が返される
context_docs = retriever.invoke(query)

# 取得されたドキュメントの数を出力
print(f"len = {len(context_docs)}")

# 最も類似度が高い（1番目の）ドキュメントを取得
first_doc = context_docs[0]

# ドキュメントのメタデータ（ファイル名、パス等）を出力
print(f"metadata = {first_doc.metadata}")

# ドキュメントの本文内容を出力
print(first_doc.page_content)

len = 4
metadata = {'file_type': '.md', 'source': 'README.md', 'file_path': 'README.md', 'file_name': 'README.md'}
- **[Deep Agents](https://github.com/langchain-ai/deepagents)** — Build agents that can plan, use subagents, and leverage file systems for complex tasks
- **[LangGraph](https://docs.langchain.com/oss/python/langgraph/overview)** — Build agents that can reliably handle complex tasks with our low-level agent orchestration framework
- **[Integrations](https://docs.langchain.com/oss/python/integrations/providers/overview)** — Chat & embedding models, tools & toolkits, and more
- **[LangSmith](https://www.langchain.com/langsmith)** — Agent evals, observability, and debugging for LLM apps
- **[LangSmith Deployment](https://docs.langchain.com/langsmith/deployments)** — Deploy and scale agents with a purpose-built platform for long-running, stateful workflows

## Why use LangChain?

LangChain helps developers build applications powered by LLMs through a standard interface for mode

このコードの役割:

1. **ベクトルデータベースの構築**:
   - 分割されたドキュメントをベクトル化してChromaに保存

2. **意味検索（Semantic Search）**:
   - クエリと意味的に近いドキュメントを検索
   - キーワード一致ではなく、意味の類似性で検索

3. **RAG（Retrieval-Augmented Generation）の基盤**:
   - 検索されたドキュメントを文脈としてLLMに渡すことで、正確な回答を生成できる

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# プロンプトテンプレートの定義
# {context}: 検索されたドキュメントの内容が挿入される
# {question}: ユーザーの質問が挿入される
prompt = ChatPromptTemplate.from_template('''\
以下の文脈だけを踏まえて質問に解答してください。

文脈"""
{context}
"""

質問:{question}
''')

# ChatGPTモデルの初期化
# gpt-4o-mini: 軽量で高速なモデル
# temperature=0: 出力の一貫性を最大化（ランダム性を最小化）
model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

# RAGチェーンの構築（パイプライン処理）
chain = (
    # ステップ1: 入力の準備
    # - context: retrieverを使ってクエリに関連するドキュメントを検索
    # - question: 元のクエリをそのまま渡す
    {"context": retriever, "question": RunnablePassthrough()}

    # ステップ2: プロンプトの生成
    # 検索されたcontextと質問をテンプレートに挿入
    | prompt

    # ステップ3: LLMで回答を生成
    | model

    # ステップ4: 出力を文字列に変換
    | StrOutputParser()
)

# チェーンを実行してクエリに対する回答を取得
output = chain.invoke(query)

# 生成された回答を出力
print(output)

LangChainを利用する理由は以下の通りです：

1. **リアルタイムデータの拡張**: 多様なデータソースや外部/内部システムに簡単に接続でき、豊富な統合ライブラリを活用できます。
2. **モデルの相互運用性**: モデルを入れ替えながら最適な選択を見つけることができ、業界の進展に迅速に適応できます。
3. **迅速なプロトタイピング**: モジュール式のコンポーネントアーキテクチャにより、アプリケーションを迅速に構築・反復できます。
4. **生産準備が整った機能**: 監視、評価、デバッグのための統合があり、信頼性の高いアプリケーションを展開できます。
5. **活発なコミュニティとエコシステム**: 統合、テンプレート、コミュニティ貢献のコンポーネントを活用し、最新のAI開発に常に追いつくことができます。
6. **柔軟な抽象化レイヤー**: 高レベルのチェーンから低レベルのコンポーネントまで、ニーズに応じたレベルで作業できます。

これらの特徴により、LangChainはLLM（大規模言語モデル）を活用したアプリケーションの開発を容易にし、迅速かつ効率的に行うことができます。


このコードの役割:

### **RAG（Retrieval-Augmented Generation）の完全な実装**

1. **検索（Retrieval）**:
   - `retriever`がクエリに関連するドキュメントを検索

2. **拡張（Augmentation）**:
   - 検索結果をプロンプトの`context`に挿入

3. **生成（Generation）**:
   - LLMが文脈を踏まえて回答を生成

### **チェーンの処理フロー:**
```
query → retriever（検索） → prompt（プロンプト生成）
      → model（LLMで回答生成） → StrOutputParser（文字列化） → output
```

### **利点:**
- ✅ **正確性**: 検索されたドキュメントに基づいて回答（幻覚を軽減）
- ✅ **透明性**: どの文脈を使ったか追跡可能
- ✅ **最新性**: ドキュメントを更新すれば最新情報で回答可能


# LangChain RAGチェーンの `{"context": retriever, "question": RunnablePassthrough()}` 詳細解説

## 基本的な仕組み

この辞書は、**プロンプトテンプレートの変数に何を渡すか**を定義しています。
```python
{"context": retriever, "question": RunnablePassthrough()}
```

プロンプトテンプレートには2つの変数があります：
```python
prompt = ChatPromptTemplate.from_template('''\
以下の文脈だけを踏まえて質問に解答してください。

文脈"""
{context}    ← この変数
"""

質問:{question}  ← この変数
''')
```

---

## 各要素の詳細説明

### 1. `"context": retriever`

**意味**: `{context}`変数には、`retriever`の結果を入れる

**実際の動き**:
```python
query = "Langchainを利用する理由は何ですか？"

# chain.invoke(query)が実行されると...

# ① retrieverがqueryを受け取る
context_docs = retriever.invoke(query)  # 関連ドキュメントを検索

# ② 検索結果がcontextに代入される
# context = "LangChainは...（ドキュメント1の内容）\n\nLangChainを使うと...（ドキュメント2の内容）..."
```

### 2. `"question": RunnablePassthrough()`

**意味**: `{question}`変数には、入力をそのまま渡す

**実際の動き**:
```python
query = "Langchainを利用する理由は何ですか？"

# RunnablePassthrough()は入力をそのまま通す
# question = "Langchainを利用する理由は何ですか？"（queryがそのまま）
```

---

## なぜ RunnablePassthrough() が必要？

### ❌ こう書けない理由:
```python
{"context": retriever, "question": query}
# ↑ queryは変数なので、チェーン定義時に固定されてしまう
```

### ✅ 正しい書き方:
```python
{"context": retriever, "question": RunnablePassthrough()}
# ↑ チェーン実行時に動的に入力を受け取る
```

---

## 具体的な実行フロー
```python
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

output = chain.invoke("Langchainを利用する理由は何ですか？")
```

### ステップバイステップ:

**Step 1**: `chain.invoke(query)` が実行される
```
入力: "Langchainを利用する理由は何ですか？"
```

**Step 2**: 辞書の各キーが処理される
```python
{
    "context": retriever.invoke("Langchainを利用する理由は何ですか？"),
    # → [Document1, Document2, Document3, ...]を検索
    
    "question": RunnablePassthrough().invoke("Langchainを利用する理由は何ですか？")
    # → "Langchainを利用する理由は何ですか？" をそのまま通す
}
```

**Step 3**: 結果が辞書になる
```python
{
    "context": "LangChainは大規模言語モデル...(Document1)\n\nLangChainを使うことで...(Document2)...",
    "question": "Langchainを利用する理由は何ですか？"
}
```

**Step 4**: この辞書がプロンプトテンプレートに渡される
```
以下の文脈だけを踏まえて質問に解答してください。

文脈"""
LangChainは大規模言語モデル...(検索結果)
"""

質問:Langchainを利用する理由は何ですか？
```

---

## より分かりやすい書き方（等価）

もし混乱するなら、こう書くこともできます：
```python
def format_docs(docs):
    """ドキュメントのリストを文字列に変換"""
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {
        "context": retriever | format_docs,  # 検索 → 文字列化
        "question": RunnablePassthrough()     # 質問をそのまま
    }
    | prompt
    | model
    | StrOutputParser()
)
```

---

## まとめ

| 要素 | 役割 | 入力 | 出力 |
|------|------|------|------|
| `retriever` | 関連ドキュメントを検索 | クエリ文字列 | Documentのリスト |
| `RunnablePassthrough()` | 入力をそのまま通す | クエリ文字列 | クエリ文字列 |

この仕組みにより、**チェーンを定義する時点では具体的なクエリを指定せず、実行時に動的に処理できる**ようになっています。

---

## 全体のデータフロー図
```
入力クエリ
    ↓
┌───────────────────────────────────┐
│ {"context": retriever,            │
│  "question": RunnablePassthrough()}│
└───────────────────────────────────┘
    ↓                    ↓
retriever          RunnablePassthrough
    ↓                    ↓
検索結果           元のクエリ
(Document[])       (文字列)
    ↓                    ↓
    └────────┬───────────┘
             ↓
        prompt (テンプレート適用)
             ↓
        model (LLMで生成)
             ↓
     StrOutputParser (文字列化)
             ↓
          最終回答
```

#総括
# LangChain RAG実装の総括

## 概要

このコードは、**RAG（Retrieval-Augmented Generation）**を実装したものです。RAGは、大規模言語モデル（LLM）に外部知識を与えることで、より正確で信頼性の高い回答を生成する手法です。

---

## 実装の全体フロー
```
1. データ収集
   ↓
2. テキスト分割
   ↓
3. ベクトル化・保存
   ↓
4. 検索（Retrieval）
   ↓
5. 回答生成（Generation）
```

---

## ステップ別詳細解説

### 1. データ収集（GitHubからMarkdownファイルを取得）
```python
from langchain_community.document_loaders import GitLoader

def file_filter(file_path: str) -> bool:
    """Markdownファイルのみをフィルタリング"""
    return file_path.endswith(".md")

loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="master",
    file_filter=file_filter,
)

raw_docs = loader.load()
print(len(raw_docs))
```

**目的**:
- LangChainの公式リポジトリからMarkdownドキュメントを取得
- これが知識ベース（Knowledge Base）となる

**出力**:
- `raw_docs`: Documentオブジェクトのリスト

---

### 2. テキスト分割（チャンキング）
```python
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=1455,      # 1チャンクの最大文字数
    chunk_overlap=0       # チャンク間の重複（推奨は10-20%）
)

docs = text_splitter.split_documents(raw_docs)
print(len(docs))
```

**目的**:
- 長いドキュメントを小さなチャンク（断片）に分割
- LLMのコンテキストウィンドウに収まるサイズにする
- 検索精度を向上させる

**推奨設定**:
```python
chunk_size=1000        # 一般的な設定
chunk_overlap=200      # chunk_sizeの10-20%
```

**出力**:
- `docs`: 分割されたDocumentオブジェクトのリスト

---

### 3. Embedding（ベクトル化）の準備
```python
from langchain_openai import OpenAIEmbeddings
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
```

**目的**:
- テキストを数値ベクトル（1536次元）に変換
- 意味的に似たテキストは近いベクトルになる

**モデル比較**:
| モデル | 次元数 | 特徴 |
|--------|--------|------|
| text-embedding-3-small | 1536 | 軽量・高速・コスパ良 |
| text-embedding-3-large | 3072 | 高精度 |

---

### 4. ベクトルデータベースの構築と検索
```python
from langchain_chroma import Chroma

# ベクトルデータベースの作成
db = Chroma.from_documents(docs, embeddings)

# Retrieverとして使用
retriever = db.as_retriever()

# クエリに関連するドキュメントを検索
query = "Langchainを利用する理由は何ですか？"
context_docs = retriever.invoke(query)

print(f"len = {len(context_docs)}")
first_doc = context_docs[0]
print(f"metadata = {first_doc.metadata}")
print(first_doc.page_content)
```

**目的**:
- ドキュメントをベクトル化してChromaに保存
- クエリに意味的に類似したドキュメントを検索

**検索の仕組み**:
1. クエリをベクトル化
2. ベクトル空間で最も近いドキュメントを検索（コサイン類似度など）
3. デフォルトで上位4件を返す

---

### 5. RAGチェーンの構築と実行
```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# プロンプトテンプレート
prompt = ChatPromptTemplate.from_template('''\
以下の文脈だけを踏まえて質問に解答してください。

文脈"""
{context}
"""

質問:{question}
''')

# LLMモデル
model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

# RAGチェーン
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

# 実行
output = chain.invoke(query)
print(output)
```

**チェーンの処理フロー**:
```
入力: "Langchainを利用する理由は何ですか？"
    ↓
① {"context": retriever, "question": RunnablePassthrough()}
   - retriever: クエリに関連するドキュメントを検索
   - RunnablePassthrough: クエリをそのまま通す
    ↓
   結果: {
     "context": "検索されたドキュメントの内容...",
     "question": "Langchainを利用する理由は何ですか？"
   }
    ↓
② prompt
   - テンプレートにcontextとquestionを挿入
    ↓
   結果: "以下の文脈だけを踏まえて...（完成したプロンプト）"
    ↓
③ model
   - GPT-4o-miniで回答を生成
    ↓
   結果: AIMessage("LangChainを利用する理由は...")
    ↓
④ StrOutputParser()
   - AIMessageを文字列に変換
    ↓
出力: "LangChainを利用する理由は..."
```

---

## RAGの利点

| 従来のLLM | RAG |
|-----------|-----|
| 訓練データのみに依存 | 外部知識を活用 |
| 幻覚（Hallucination）が起きやすい | 事実に基づく回答 |
| 最新情報に弱い | ドキュメント更新で最新化可能 |
| 知識の根拠が不明 | 参照元を追跡可能 |

---

## 主要コンポーネント

### データローダー
- **GitLoader**: GitHubリポジトリからドキュメントを取得

### テキスト分割
- **CharacterTextSplitter**: 文字数ベースで分割
- **RecursiveCharacterTextSplitter**: 段落・文の境界を考慮（推奨）

### Embeddings
- **OpenAIEmbeddings**: テキストをベクトルに変換

### ベクトルデータベース
- **Chroma**: 軽量なベクトルDB（ローカル実行可能）
- その他: Pinecone, Weaviate, Qdrant など

### LLM
- **ChatOpenAI**: OpenAIのChatGPTモデル

### チェーン構築
- **RunnablePassthrough**: 入力をそのまま渡す
- **StrOutputParser**: LLMの出力を文字列に変換

---

## パイプライン演算子 `|` の意味
```python
A | B | C
```

- `A`の出力 → `B`の入力
- `B`の出力 → `C`の入力
- Unix のパイプと同じ概念

---

## 改善の余地

### 1. chunk_overlapの追加
```python
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200  # 文脈の連続性を保つ
)
```

### 2. より高度な分割方法
```python
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", "。", ".", " ", ""]
)
```

### 3. 検索結果の数を調整
```python
retriever = db.as_retriever(search_kwargs={"k": 5})  # 上位5件
```

### 4. ハイブリッド検索
```python
# ベクトル検索 + キーワード検索の組み合わせ
retriever = db.as_retriever(
    search_type="mmr",  # Maximum Marginal Relevance
    search_kwargs={"k": 5, "fetch_k": 20}
)
```

### 5. 引用元の表示
```python
for doc in context_docs:
    print(f"ソース: {doc.metadata['source']}")
    print(f"内容: {doc.page_content[:200]}...")
```

---

## 完全なコード（まとめ）
```python
from langchain_community.document_loaders import GitLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import os
from google.colab import userdata

# 1. APIキーの設定
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 2. データ収集
loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="master",
    file_filter=lambda x: x.endswith(".md"),
)
raw_docs = loader.load()

# 3. テキスト分割
text_splitter = CharacterTextSplitter(chunk_size=1455, chunk_overlap=0)
docs = text_splitter.split_documents(raw_docs)

# 4. Embedding
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 5. ベクトルデータベース構築
db = Chroma.from_documents(docs, embeddings)
retriever = db.as_retriever()

# 6. プロンプト・モデル・チェーン
prompt = ChatPromptTemplate.from_template('''\
以下の文脈だけを踏まえて質問に解答してください。

文脈"""
{context}
"""

質問:{question}
''')

model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

# 7. 実行
query = "Langchainを利用する理由は何ですか？"
output = chain.invoke(query)
print(output)
```

---

## まとめ

この実装は、以下の技術を組み合わせたRAGシステムです：

1. **データ収集**: GitHubからMarkdownドキュメントを取得
2. **前処理**: テキストを適切なサイズに分割
3. **ベクトル化**: テキストを意味ベクトルに変換
4. **検索**: クエリに関連するドキュメントを取得
5. **生成**: 検索結果を文脈としてLLMで回答を生成

これにより、LLMが持っていない最新の知識や特定のドメイン知識を使って、正確で信頼性の高い回答を生成できます。